In [1]:
import os
os.environ["KERAS_BACKEND"] = "torch"

import keras
from keras.layers import Dense, Input, Dropout, Conv2D, MaxPooling2D, Flatten
from keras.models import Sequential
from keras.losses import SparseCategoricalCrossentropy
from keras.optimizers import Adam

import numpy as np
import cv2

from sklearn.model_selection import train_test_split

from matplotlib import pyplot as plt

In [2]:
print(f"Backend : {keras.backend.backend()}")

Backend : torch


In [17]:
plus_model = Sequential()
plus_model.add(Input((128, 128, 3)))
plus_model.add(Conv2D(32, (3, 3), activation="relu"))
plus_model.add(MaxPooling2D((2, 2)))
plus_model.add(Conv2D(64, (3, 3), activation="relu"))
plus_model.add(MaxPooling2D((2, 2)))
plus_model.add(Conv2D(128, (3, 3), activation="relu"))
plus_model.add(MaxPooling2D((2, 2)))
plus_model.add(Dropout(0.5))
plus_model.add(Flatten())
plus_model.add(Dense(128, activation="relu"))
plus_model.add(Dense(64, activation="relu"))
plus_model.add(Dense(32, activation="relu"))
plus_model.add(Dense(2)) # male | female

plus_model.compile(optimizer=Adam(), loss=SparseCategoricalCrossentropy(), metrics=["accuracy"])

In [6]:
plus_model.summary()


Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_16 (Conv2D)              │ (None, 126, 126, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_16 (MaxPooling2D) │ (None, 63, 63, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_17 (Conv2D)              │ (None, 61, 61, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_17 (MaxPooling2D) │ (None, 30, 30, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_18 (Conv2D)              │ (None, 28, 28, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_18 (MaxPooling2D) │ (None, 14, 14, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_10 (Dropout)            │ (None, 14, 14, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_5 (Flatten)             │ (None, 25088)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_23 (Dense)                │ (None, 128)            │     3,211,392 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_24 (Dense)                │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_25 (Dense)                │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_26 (Dense)                │ (None, 2)              │            66 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 9,945,128 (37.94 MB)

 Trainable params: 3,315,042 (12.65 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 6,630,086 (25.29 MB)

In [6]:
baseline_model = Sequential()
baseline_model.add(Input((128, 128, 3)))
baseline_model.add(Conv2D(16, (3, 3), activation="relu"))
baseline_model.add(MaxPooling2D((2, 2)))
baseline_model.add(Conv2D(32, (3, 3), activation="relu"))
baseline_model.add(MaxPooling2D((2, 2)))
baseline_model.add(Dropout(0.45))
baseline_model.add(Flatten())
baseline_model.add(Dense(64, activation="relu"))
baseline_model.add(Dense(32, activation="relu"))
baseline_model.add(Dense(16, activation="relu"))
baseline_model.add(Dense(2))

baseline_model.compile(optimizer=Adam(), loss=SparseCategoricalCrossentropy(), metrics=["accuracy"])

In [7]:
baseline_model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_4 (Conv2D)               │ (None, 126, 126, 16)   │           448 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_4 (MaxPooling2D)  │ (None, 63, 63, 16)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 61, 61, 32)     │         4,640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_5 (MaxPooling2D)  │ (None, 30, 30, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 30, 30, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_2 (Flatten)             │ (None, 28800)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 64)             │     1,843,264 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 2)              │            34 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,850,994 (7.06 MB)

 Trainable params: 1,850,994 (7.06 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
# training plus

import torch
import gc

data = os.listdir("UTKFace")
labels = []

for i in data:
    labels.append(i.split("_")[1])

X_train, X_test, y_train, y_test = train_test_split(data, labels, stratify=labels, random_state=42, test_size=0.2)


def run_test():
    global X_test, y_test
    X = []

    for i in X_test:
        img = cv2.imread(f"UTKFace/{i}")
        img = cv2.resize(img, (128, 128))
        X.append(img)

    X = np.array(X)
    y_test = np.array(y_test)

    pred = plus_model.predict(X)

    pred_classes = np.argmax(pred, axis=1)

    corrects = (pred_classes == y_test.astype(int)).sum()
    return corrects / len(X_test) * 100



batch_size = 512
remaining = len(X_train)

index = 0

epochs = 80

history = []

for epoch in range(epochs):
    while remaining != 0:
        if remaining > batch_size:
            batch_size = batch_size
        else :
            batch_size = remaining

        X = []
        y = []

        for i in range(batch_size):
            img = cv2.imread(f"UTKFace/{X_train[index]}")
            img = cv2.resize(img, (128, 128))

            label = int(X_train[index].split("_")[1])

            X.append(img)
            y.append(label)
            index = index + 1

        X = np.array(X)
        y = np.array(y)

        plus_model.train_on_batch(X, y)
        remaining -= batch_size

        gc.collect()
        torch.cuda.empty_cache()
        del X, y

    remaining = len(X_train)
    batch_size = 512
    index = 0
    test_res = run_test()
    history.append(test_res)
    print(f"Epoch {epoch+1}/80 done; test accuracy : {test_res}%")
    if test_res >= 95:
        print("95% Accuracy, training finished")
        break

149/149 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step
Epoch 1/80 done; test accuracy : 66.59637283846477%
149/149 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step
Epoch 2/80 done; test accuracy : 71.9105862505272%
149/149 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step
Epoch 3/80 done; test accuracy : 80.07169970476592%
149/149 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step
Epoch 4/80 done; test accuracy : 78.4479122733024%
149/149 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step
Epoch 5/80 done; test accuracy : 83.53015605229861%
149/149 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step
Epoch 6/80 done; test accuracy : 80.89413749472796%
149/149 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step
Epoch 7/80 done; test accuracy : 83.69886123998313%
149/149 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step
Epoch 8/80 done; test accuracy : 71.65752846900043%
149/149 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step
Epoch 9/80 done; test accuracy : 76.44453816954872%
149/149 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step
Epoch 10/80 done; test accuracy : 82.49683677773092%
149/149 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step
Epoch 11/80 done; test accurac

In [8]:
# training baseline

import torch
import gc

data = os.listdir("UTKFace")
labels = []

for i in data:
    labels.append(i.split("_")[1])

X_train, X_test, y_train, y_test = train_test_split(data, labels, stratify=labels, random_state=42, test_size=0.2)

# taking some of the training data from baseline

X_train, _, y_train, _ = train_test_split(X_train, y_train, stratify=y_train, random_state=42, test_size=0.3)

def run_test():
    global X_test, y_test
    X = []

    for i in X_test:
        img = cv2.imread(f"UTKFace/{i}")
        img = cv2.resize(img, (128, 128))
        X.append(img)

    X = np.array(X)
    y_test = np.array(y_test)

    pred = baseline_model.predict(X)

    pred_classes = np.argmax(pred, axis=1)

    corrects = (pred_classes == y_test.astype(int)).sum()
    return corrects / len(X_test) * 100



batch_size = 512
remaining = len(X_train)

index = 0

epochs = 50

history = []

for epoch in range(epochs):
    while remaining != 0:
        if remaining > batch_size:
            batch_size = batch_size
        else :
            batch_size = remaining

        X = []
        y = []

        for i in range(batch_size):
            img = cv2.imread(f"UTKFace/{X_train[index]}")
            img = cv2.resize(img, (128, 128))

            label = int(X_train[index].split("_")[1])

            X.append(img)
            y.append(label)
            index = index + 1

        X = np.array(X)
        y = np.array(y)

        baseline_model.train_on_batch(X, y)
        remaining -= batch_size

        gc.collect()
        torch.cuda.empty_cache()
        del X, y

    remaining = len(X_train)
    batch_size = 512
    index = 0
    test_res = run_test()
    history.append(test_res)
    print(f"Epoch {epoch+1}/50 done; test accuracy : {test_res}%")
    if test_res >= 95:
        print("95% Accuracy, training finished")
        break

149/149 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step
Epoch 1/50 done; test accuracy : 47.74356811471952%
149/149 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step
Epoch 2/50 done; test accuracy : 59.53184310417545%
149/149 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step
Epoch 3/50 done; test accuracy : 68.0936313791649%
149/149 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step
Epoch 4/50 done; test accuracy : 73.82960776043863%
149/149 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step
Epoch 5/50 done; test accuracy : 75.09489666807254%
149/149 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step
Epoch 6/50 done; test accuracy : 75.74862927035007%
149/149 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step
Epoch 7/50 done; test accuracy : 79.88190636862083%
149/149 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step
Epoch 8/50 done; test accuracy : 70.5187684521299%
149/149 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step
Epoch 9/50 done; test accuracy : 81.75875158161114%
149/149 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step
Epoch 10/50 done; test accuracy : 83.15056938000843%
149/149 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step
Epoch 11/50 done; test accurac

In [3]:
import os

def load_and_pre_proccess(path):
    if os.path.exists(path):
        img = cv2.imread(path)
        img = cv2.resize(img, (128, 128))

        return img
    else:
        print(f"Image on path, {path} not found")

In [1]:
import cv2
import numpy as np
import os

os.environ["KERAS_BACKEND"] = "torch"
import keras

# Load the model
plus_model = keras.models.load_model("gender-model-plus.keras")
baseline_model = keras.models.load_model("gender-model-baseline.keras")

# Face detector
face_cascade = cv2.CascadeClassifier(
    cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
)

IMG_SIZE = (128, 128)

def preprocess(face_bgr):
    face = cv2.resize(face_bgr, IMG_SIZE)
    face = face.reshape(1, 128, 128, 3)
    return face

# Open webcam
cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()
    if not ret:
        break

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    faces = face_cascade.detectMultiScale(gray, 1.1, 4, minSize=(50, 50))

    for (x, y, w, h) in faces:
        face_img = frame[y:y+h, x:x+w]
        features = preprocess(face_img)

        pred = plus_model.predict(features, verbose=0)
        pred_class = np.argmax(pred[0])
        pred2 = baseline_model.predict(features, verbose=0)
        pred_class2 = np.argmax(pred2[0])


        label = "male" if pred_class == 0 else "female"
        label2 = "male" if pred_class2 == 0 else "female"
        color = (255, 0, 0) if label == "male" else (0, 0, 255)

        cv2.rectangle(frame, (x, y), (x+w, y+h), color, 2)
        cv2.putText(frame, label, (x, y-10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, color, 2)
        cv2.putText(frame, label2, (x, y-30),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)


    cv2.imshow("OpenMTS Gender Live Test", frame)
    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

cap.release()
cv2.destroyAllWindows()

In [19]:
plus_model.save("gender-model-plus.keras")

In [16]:
baseline_model.save("gender-model-baseline.keras")

In [4]:
import os

size_bytes = os.path.getsize('gender-model-plus.keras')
size_mb = size_bytes / (1024 * 1024)
print(f"Size: {size_mb:.2f} MB")

Size: 37.99 MB


In [17]:
import os

size_bytes = os.path.getsize('gender-model-baseline.keras')
size_mb = size_bytes / (1024 * 1024)
print(f"Size: {size_mb:.2f} MB")

Size: 21.23 MB
